# Aggregated signal analysis

This notebook performs a statistical analysis of the aggregated seismic acceleration signal, obtained by combining all 2,304,000 samples from the 48 truncated signals into a single time series. The analysis mirrors the structure of notebook 03 but treats the entire aggregated dataset as a single signal, and compares results across distance groups (Near, Mid, Far).

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import logging
import matplotlib.pyplot as plt
from src.plot_settings import set_plot_style
from src.signals_ import (
    plot_empirical_pdfs,
    gaussian_fit_analysis,
    heavy_tail_assessment,
    compute_moment_scaling,
    compute_scaling_exponents,
    test_scaling_linearity,
    fit_piecewise_scaling,
    compute_displacement_autocorrelation,
    analyze_autocorrelation_scaling,
    plot_pdfs_by_group,
    gaussian_fit_by_group,
    heavy_tail_by_group
)
from src.cleaning_metadata import clean_metadata
from src.io import build_metadata

colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")
logger.info("Libraries imported successfully")

## 2. Data loading

Preprocessed aggregated acceleration data are loaded from the parquet file 
produced in notebook 02. Only signals with at least 48000 samples are included, 
all truncated to exactly 48000 samples. Each signal has been baseline-corrected 
and normalized by its standard deviation, so that $\bar{a} = 0$ and $\sigma_a = 1$.

In [7]:
# Load preprocessed data
df_acc_agg = pd.read_parquet('../data/processed/acc_preprocessed_aggregated.parquet')
print("Shape:", df_acc_agg.shape)
print(df_acc_agg.head())

Shape: (2304000, 4)
                                       file  sample  acceleration  \
0  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       0 -6.666667e-10   
1  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       1 -6.666667e-10   
2  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       2 -6.666667e-10   
3  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       3 -6.666667e-10   
4  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       4 -6.666667e-10   

   acceleration_normalized  
0            -3.401661e-08  
1            -3.401661e-08  
2            -3.401661e-08  
3            -3.401661e-08  
4            -3.401661e-08  


In [8]:
# Figures output directory
FIGURES_DIR = Path('../figures/04_aggregated')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 3. Distance groups

Stations are assigned to distance groups based on the epicentral distance, 
following the same grouping defined in notebook 01:

- **Near**: $d \leq 48.4$ km
- **Mid**: $53.5 \leq d \leq 78.5$ km
- **Far**: $80.9 \leq d \leq 109.5$ km

The grouping is retrieved from the preprocessed metadata and merged 
into the aggregated acceleration dataframe.

In [9]:
# Load metadata to retrieve distance groups
zip_path = Path("..") / "data" / "raw" / "query.zip"
df_meta = build_metadata(zip_path)
df_meta_clean = clean_metadata(df_meta)

# Assign distance groups
df_stations = df_meta_clean.drop_duplicates('STATION_CODE')[['STATION_CODE', 'EPICENTRAL_DISTANCE_KM']].copy()
df_stations['DISTANCE_GROUP'] = pd.qcut(
    df_stations['EPICENTRAL_DISTANCE_KM'],
    q=3,
    labels=['Near', 'Mid', 'Far']
)

# Extract station code from file name and merge
df_acc_agg['STATION_CODE'] = df_acc_agg['file'].apply(lambda x: x.split('.')[1])
df_acc_agg = df_acc_agg.merge(
    df_stations[['STATION_CODE', 'DISTANCE_GROUP']],
    on='STATION_CODE',
    how='left'
)

groups = ['Near', 'Mid', 'Far']
group_colors = [colors[0], colors[1], colors[2]]

print(df_acc_agg['DISTANCE_GROUP'].value_counts())

# Create single global signal and per-group signals
# Global: all 2,304,000 samples as one signal
df_global = df_acc_agg.copy()
df_global['file'] = 'XX.ALL.00.HNA'

# Per group: each group as one signal
df_near = df_acc_agg[df_acc_agg['DISTANCE_GROUP'] == 'Near'].copy()
df_near['file'] = 'XX.NEAR.00.HNA'

df_mid = df_acc_agg[df_acc_agg['DISTANCE_GROUP'] == 'Mid'].copy()
df_mid['file'] = 'XX.MID.00.HNA'

df_far = df_acc_agg[df_acc_agg['DISTANCE_GROUP'] == 'Far'].copy()
df_far['file'] = 'XX.FAR.00.HNA'

# Combined for by-group analysis
df_by_group = pd.concat([df_near, df_mid, df_far], ignore_index=True)

print(f"\nGlobal signal: {len(df_global):,} samples")
print(f"Near: {len(df_near):,} | Mid: {len(df_mid):,} | Far: {len(df_far):,}")

DISTANCE_GROUP
Far     1152000
Mid      864000
Near     288000
Name: count, dtype: int64

Global signal: 2,304,000 samples
Near: 288,000 | Mid: 864,000 | Far: 1,152,000


## 4. PDF analysis

Empirical probability density functions and distribution fits are computed 
first on the full aggregated dataset, then separately for each distance group. 
This allows both a global characterization of the acceleration distribution 
and a comparison of its statistical properties across distance groups.

### 4.1 Aggregated empirical PDF

Empirical PDF of the full aggregated dataset, combining all 48 signals.

In [10]:
# Aggregated empirical PDF — global signal
plot_empirical_pdfs(df_global, bins=100, log_scale=True, output_dir=FIGURES_DIR / 'pdf')

Saved: 1/1 PDFs
All PDFs saved successfully!


### 4.2 Gaussian fit and normality assessment

A Gaussian distribution $\mathcal{N}(\mu, \sigma^2)$ is fitted to the 
full aggregated signal. The Anderson-Darling test is used to formally 
assess normality, and kurtosis and skewness are computed as quantitative 
measures of the shape of the distribution:

$$\kappa = \frac{\mu_4}{\sigma^4} - 3, \qquad \gamma = \frac{\mu_3}{\sigma^3}$$

A Q-Q plot against the Gaussian distribution is produced to visualize 
deviations from normality.

In [11]:
# Gaussian fit — global signal
df_gaussian_results_agg = gaussian_fit_analysis(df_global, bins=100, log_scale=True, output_dir=FIGURES_DIR / 'gaussian_fit')

Saved: 1/1 individual plots
All individual plots saved successfully!
Non-Gaussian signals (AD test, p<5%): 1/1


In [12]:
# Display Gaussian fit results
pd.set_option('display.max_rows', None)
display(df_gaussian_results_agg[['station', 'stream', 'kurtosis', 'skewness', 'ad_statistic', 'ad_critical_5pct', 'non_gaussian']])
pd.reset_option('display.max_rows')

,station,stream,kurtosis,skewness,ad_statistic,ad_critical_5pct,non_gaussian
0,ALL,HNA,57.5772,-0.0851,439425.8854,0.787,True


In [13]:
# Save Gaussian fit results
try:
    df_gaussian_results_agg.to_parquet('../data/processed/gaussian_fit_results_agg.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 4.3 Heavy-tail assessment

Gaussian, Laplace, Student-t and Lévy stable distributions are fitted 
to the full aggregated signal. The best fit is selected using the Akaike 
Information Criterion:

$$\text{AIC} = 2k - 2\ln\hat{L}$$

The power law exponent is estimated using the Hill estimator on the 
top 10\% of values:

$$\hat{\alpha} = \left( \frac{1}{k} \sum_{i=1}^{k} \ln \frac{x_{(i)}}{x_{(k+1)}} \right)^{-1}$$

In [ ]:
# Check partial results if available
try:
    df_partial = pd.read_parquet('../data/processed/heavy_tail_aggregated_partial.parquet')
    print(f"Partial results available (aggregated): {len(df_partial)} signals already processed")
except Exception:
    print("No partial results found (aggregated), starting from scratch")

In [ ]:
# Heavy-tail assessment — global signal
df_heavy_tail_agg = heavy_tail_assessment(df_global,
                                           output_dir=FIGURES_DIR / 'heavy_tail',
                                           resume=True)
try:
    df_heavy_tail_agg.to_parquet('../data/processed/heavy_tail_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

In [ ]:
print(df_heavy_tail_agg.columns.tolist())

### 4.4 PDF comparison across distance groups

Empirical PDFs are plotted for each distance group on the same axes 
to visually compare the shape of the acceleration distribution across 
Near, Mid and Far stations.

In [ ]:
# PDF comparison across distance groups — one curve per group
plot_pdfs_by_group(df_by_group, groups, group_colors, output_dir=FIGURES_DIR)

### 4.5 Gaussian fit comparison across distance groups

Gaussian fits and normality assessments are performed separately for 
each distance group. Kurtosis, skewness and Anderson-Darling statistics 
are compared to investigate whether deviations from Gaussianity change 
with epicentral distance.

In [ ]:
# Gaussian fit by group — one signal per group
df_gaussian_by_group = gaussian_fit_by_group(df_by_group, groups,
                                              output_dir=FIGURES_DIR)
try:
    df_gaussian_by_group.to_parquet('../data/processed/gaussian_fit_by_group.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### 4.6 Heavy-tail comparison across distance groups

Distribution fits and CCDF analysis are performed separately for each 
distance group to investigate whether the tail behavior changes with 
epicentral distance. In particular, the Student-t degrees of freedom $\nu$ 
and the power law exponent $\hat{\alpha}$ are compared across groups — 
lower values indicate heavier tails.

In [ ]:
# Check partial results if available
try:
    df_partial_group = pd.read_parquet('../data/processed/heavy_tail_by_group_partial.parquet')
    print(f"Partial results available (by group): {len(df_partial_group)} groups already processed")
except Exception:
    print("No partial results found (by group), starting from scratch")

In [ ]:
# Heavy-tail by group — one signal per group
df_heavy_tail_by_group = heavy_tail_by_group(df_by_group, groups,
                                              output_dir=FIGURES_DIR)
try:
    df_heavy_tail_by_group.to_parquet('../data/processed/heavy_tail_by_group.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

## 5. Moment scaling analysis

The scaling behavior of statistical moments is investigated to detect 
signatures of anomalous or strongly anomalous diffusion. The analysis 
is performed first on the full aggregated dataset, then compared across 
distance groups. For each signal, the $q$-th order moment is computed 
at different time scales $\tau$:

$$M_q(\tau) = \langle |a|^q \rangle_\tau$$

If the process exhibits scaling, the moments obey a power law:

$$M_q(\tau) \sim \tau^{\zeta(q)}$$

For normal diffusion, $\zeta(q) = q/2$ (linear in $q$). Deviations from 
linearity indicate anomalous or strongly anomalous diffusion.

### 5.1 Computation of q-th order moments

Moments are computed for moment orders $q \in \{0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0\}$ and time scales $\tau \in \{10, 50, 100, 200, 500, 1000, 2000, 5000, 10000\}$ samples.

In [ ]:
q_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Compute moments — global signal
df_moments_agg = compute_moment_scaling(df_global, q_values, tau_values)
print("Shape:", df_moments_agg.shape)

try:
    df_moments_agg.to_parquet('../data/processed/moments_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### 5.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

A plot of $\zeta(q)$ vs $q$ is produced for each signal, with the reference 
line $\zeta(q) = q/2$ corresponding to normal diffusion.

In [ ]:
# Compute scaling exponents on full aggregated dataset
df_exponents_agg = compute_scaling_exponents(df_moments_agg,
                                              output_dir=FIGURES_DIR / 'scaling')
try:
    df_exponents_agg.to_parquet('../data/processed/scaling_exponents_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### 5.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing a linear 
and a quadratic fit using AIC. A piecewise linear fit is also performed 
to detect the presence of two distinct scaling regimes:

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes.

In [ ]:
# Linearity check on full aggregated dataset
df_linearity_agg = test_scaling_linearity(df_exponents_agg,
                                           output_dir=FIGURES_DIR / 'scaling')
try:
    df_linearity_agg.to_parquet('../data/processed/scaling_linearity_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

# Piecewise linear fit on full aggregated dataset
df_piecewise_agg = fit_piecewise_scaling(df_exponents_agg,
                                          output_dir=FIGURES_DIR / 'scaling')
try:
    df_piecewise_agg.to_parquet('../data/processed/scaling_piecewise_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### 5.4 Comparison across distance groups

Moment scaling analysis is performed separately for each distance group 
to investigate whether the scaling exponents $\zeta(q)$ and the linearity 
of the scaling change with epicentral distance.

In [ ]:
df_moments_by_group = {}
df_exponents_by_group = {}
df_linearity_by_group = {}
df_piecewise_by_group = {}

for group, df_g in [('Near', df_near), ('Mid', df_mid), ('Far', df_far)]:

    # Moments
    df_moments_by_group[group] = compute_moment_scaling(df_g, q_values, tau_values)

    # Scaling exponents
    df_exponents_by_group[group] = compute_scaling_exponents(
        df_moments_by_group[group],
        output_dir=FIGURES_DIR / f'scaling_{group.lower()}'
    )

    # Linearity check
    df_linearity_by_group[group] = test_scaling_linearity(
        df_exponents_by_group[group],
        output_dir=FIGURES_DIR / f'scaling_{group.lower()}'
    )

    # Piecewise fit
    df_piecewise_by_group[group] = fit_piecewise_scaling(
        df_exponents_by_group[group],
        output_dir=FIGURES_DIR / f'scaling_{group.lower()}'
    )

    try:
        df_exponents_by_group[group].to_parquet(
            f'../data/processed/scaling_exponents_{group.lower()}.parquet', index=False)
        df_piecewise_by_group[group].to_parquet(
            f'../data/processed/scaling_piecewise_{group.lower()}.parquet', index=False)
        print(f"Saved {group} results successfully!")
    except Exception as e:
        print(f"Error saving {group} results: {e}")

# Comparison plot
fig, ax = plt.subplots(figsize=(8, 6))
for i, group in enumerate(groups):
    df_mean = df_exponents_by_group[group].groupby('q')['zeta'].mean().reset_index()
    ax.plot(df_mean['q'], df_mean['zeta'], 'o-', color=group_colors[i],
            linewidth=1.5, markersize=5, label=group)
q_arr = np.array(q_values)
ax.plot(q_arr, 0.5 * q_arr, '--', color='gray', linewidth=1,
        label='Normal diffusion ($\\zeta(q) = q/2$)')
ax.set_xlabel('$q$')
ax.set_ylabel('$\\zeta(q)$')
ax.set_title('Mean scaling exponents by distance group')
ax.legend(title='Distance group')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'scaling_comparison_by_group.pdf', bbox_inches='tight')
plt.show()

## 6. Displacement autocorrelation functions

The displacement autocorrelation function is computed for each signal 
following the framework of Vollmer et al. (2021):

$$C(t_1, t_2) = \langle (a(t_1) - a_0)(a(t_2) - a_0) \rangle$$

where $a_0 = a(0)$ is the initial value of the signal. The function 
is evaluated on a logarithmic grid of $(t_1, t_2)$ pairs. The analysis 
is performed first on the full aggregated dataset, then compared across 
distance groups.

### 6.1 Computation

Autocorrelation functions are computed on a logarithmic grid of $n = 50$ 
points for both $t_1$ and $t_2$, resulting in a $50 \times 50$ matrix 
for each signal.

In [ ]:
# Autocorrelation — global signal
df_autocorr_agg, C_matrices_agg = compute_displacement_autocorrelation(
    df_global, output_dir=FIGURES_DIR / 'autocorrelation')

try:
    df_autocorr_agg.to_parquet('../data/processed/autocorr_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### 6.2 Scaling behavior

The scaling exponent $\beta$ is estimated from the diagonal $C(t, t)$, 
which is expected to scale as:

$$C(t, t) \sim t^\beta$$

The exponent $\beta$ is estimated by linear regression of 
$\log |C(t, t)|$ vs $\log t$. A value of $\beta = 1$ is consistent 
with normal diffusion, while $\beta \neq 1$ indicates anomalous scaling.

In [ ]:
# Scaling behavior on full aggregated dataset
df_autocorr_scaling_agg = analyze_autocorrelation_scaling(
    C_matrices_agg, output_dir=FIGURES_DIR / 'autocorrelation')

try:
    df_autocorr_scaling_agg.to_parquet(
        '../data/processed/autocorr_scaling_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### 6.3 Comparison across distance groups

Displacement autocorrelation functions and scaling exponents $\beta$ 
are computed separately for each distance group to investigate whether 
the temporal correlation structure of the signals changes with epicentral 
distance.

In [ ]:
df_autocorr_by_group = {}
C_matrices_by_group = {}
df_autocorr_scaling_by_group = {}

for group, df_g in [('Near', df_near), ('Mid', df_mid), ('Far', df_far)]:

    df_autocorr_by_group[group], C_matrices_by_group[group] = \
        compute_displacement_autocorrelation(
            df_g, output_dir=FIGURES_DIR / f'autocorrelation_{group.lower()}')

    df_autocorr_scaling_by_group[group] = analyze_autocorrelation_scaling(
        C_matrices_by_group[group],
        output_dir=FIGURES_DIR / f'autocorrelation_{group.lower()}')

    try:
        df_autocorr_scaling_by_group[group].to_parquet(
            f'../data/processed/autocorr_scaling_{group.lower()}.parquet', index=False)
        print(f"Saved {group} results successfully!")
    except Exception as e:
        print(f"Error saving {group} results: {e}")

# Comparison plot
fig, ax = plt.subplots(figsize=(8, 5))
for i, group in enumerate(groups):
    df_group_scaling = df_autocorr_scaling_by_group[group]
    ax.bar(
        [j + i * 0.25 for j in range(len(df_group_scaling))],
        df_group_scaling['beta'],
        width=0.25,
        color=group_colors[i],
        label=group
    )
ax.axhline(1, color='gray', linewidth=0.8, linestyle='--',
           label='$\\beta = 1$ (normal diffusion)')
ax.set_ylabel('$\\beta$')
ax.set_title('Autocorrelation scaling exponent by distance group')
ax.legend(title='Distance group')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'autocorr_scaling_by_group.pdf', bbox_inches='tight')
plt.show()

## 7. Summary

A summary table collects the main results from all analyses, first for 
the full aggregated dataset, then for each distance group. The table 
allows a direct comparison of the statistical properties of the signals 
across distance groups.

| Column | Description |
|--------|-------------|
| `kurtosis` | Excess kurtosis $\kappa$ |
| `skewness` | Skewness $\gamma$ |
| `non_gaussian` | Anderson-Darling test result ($\alpha = 0.05$) |
| `best_fit_aic` | Best fitting distribution by AIC |
| `student_t_df` | Student-t degrees of freedom $\nu$ |
| `power_law_exp` | Power law exponent $\hat{\alpha}$ (Hill estimator) |
| `q_break` | Piecewise scaling breakpoint $q^*$ |
| `slope_low` | Scaling slope for $q \leq q^*$ |
| `slope_high` | Scaling slope for $q > q^*$ |
| `beta` | Autocorrelation scaling exponent $\beta$ |

In [ ]:
# Summary for full aggregated dataset
df_summary_agg = df_gaussian_results_agg[['station', 'stream', 'kurtosis', 'skewness', 'non_gaussian']]\
    .merge(df_heavy_tail_agg[['station', 'stream', 'best_fit_aic', 'student_t_df', 'power_law_exp']],
           on=['station', 'stream'])\
    .merge(df_piecewise_agg[['station', 'stream', 'q_break', 'slope_low', 'slope_high']],
           on=['station', 'stream'])\
    .merge(df_autocorr_scaling_agg[['station', 'stream', 'beta', 'r2']],
           on=['station', 'stream'])

print("Summary — full aggregated dataset:")
pd.set_option('display.max_rows', None)
display(df_summary_agg)
pd.reset_option('display.max_rows')

try:
    df_summary_agg.to_parquet('../data/processed/summary_aggregated.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

# Summary by distance group
df_summary_by_group = df_gaussian_by_group[['group', 'kurtosis', 'skewness', 'non_gaussian']]\
    .merge(df_heavy_tail_by_group[['group', 'best_fit', 'student_t_df', 'power_law_exp']],
           on='group')\
    .merge(
        pd.DataFrame([{
            'group': group,
            'q_break': df_piecewise_by_group[group]['q_break'].mean(),
            'slope_low': df_piecewise_by_group[group]['slope_low'].mean(),
            'slope_high': df_piecewise_by_group[group]['slope_high'].mean(),
            'beta': df_autocorr_scaling_by_group[group]['beta'].mean(),
        } for group in groups]),
        on='group'
    )

print("\nSummary — by distance group:")
display(df_summary_by_group)

try:
    df_summary_by_group.to_parquet('../data/processed/summary_by_group.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")